# P16D cube + NDVI + TempCNN (Rondonia, `samples_deforestation_rondonia_1M.rds`)

Training samples must share the **same timeline** as the prediction cube after `cube_regularize`. If you see **`.check_samples_tile_match_timeline`**, the period/resolution (or dates) do not match how the RDS was built. This notebook follows [`inst/demo-paper-2025/tempcnn_model_training.R`](../demo-paper-2025/tempcnn_model_training.R): **P16D**, **600 m**, bands including cloud + **NDVI** on B04/B08.

Training RDS is loaded by the **worker** from HTTPS (no local `readRDS`). Requires a Python client with ML helpers (e.g. [PondiB/openeo-python-client](https://github.com/PondiB/openeo-python-client)). Adjust credentials for your environment.

## 1. Connect

In [1]:
import openeo

In [2]:
connection = openeo.connect("http://127.0.0.1:8000")
connection.authenticate_basic("user", "password")

<Connection to 'http://127.0.0.1:8000/' with BasicBearerAuth>

## 2. List collections and processes

In [3]:
print("Available collections:", connection.list_collection_ids())
print("\nAvailable processes:", [p["id"] for p in connection.list_processes()])

Available collections: ['mpc-landsat-c2-l2', 'mpc-sentinel-2-l2a', 'mpc-sentinel-1-grd', 'mpc-sentinel-1-rtc', 'aws-sentinel-2-l2a', 'aws-landsat-c2-l2', 'cdse-sentinel-2-l2a']

Available processes: ['load_collection', 'mlm_class_random_forest', 'mlm_class_svm', 'mlm_class_xgboost', 'ml_class_mlp', 'mlm_class_tempcnn', 'mlm_class_tae', 'mlm_class_lighttae', 'ml_fit', 'ml_predict', 'ml_validate', 'ml_validate_kfold', 'ml_tune_grid', 'ml_tune_random', 'ml_predict_probabilities', 'ml_uncertainty_class', 'ml_smooth_class', 'ml_label_class', 'cube_regularize', 'ndvi', 'merge_cubes', 'filter_bands', 'save_result', 'load_result', 'export_cube', 'import_cube', 'export_ml_model', 'import_ml_model', 'save_ml_model', 'load_stac_ml', 'load_ml_model']


## 3. Area of interest and time range

In [4]:
bbox = {
    "west": -63.33,
    "south": -12.03,
    "east": -62.43,
    "north": -11.13,
    "crs": 4326,
}
temporal_extent = ["2022-01-01", "2022-12-31"]

## 4. Load Sentinel-2 collection

In [5]:
datacube = connection.load_collection(
    "aws-sentinel-2-l2a",
    spatial_extent=bbox,
    temporal_extent=temporal_extent
)

## 5. Regularize (`P16D`, 600 m — matches paper training script)

In [6]:
datacube = datacube.process(
    process_id="cube_regularize",
    arguments={
        "data": datacube,
        "period": "P16D",
        "resolution": 600,
    },
)

## 6. NDVI

In [7]:
datacube = datacube.ndvi(red="B04", nir="B08", target_band="NDVI")

## 7. Training samples (URL — worker downloads RDS)

In [8]:
training_set = "https://github.com/Open-Earth-Monitor/openeocraft/raw/main/inst/demo-paper-2025/data/samples_deforestation_rondonia.rds"

## 8. TempCNN model definition

In [9]:
tempcnn_model_init = connection.mlm_class_tempcnn(
    optimizer="adam",
    learning_rate=0.0005,
    seed=42,
)

## 9. Train (`ml_fit`)

In [10]:
tempcnn_model = tempcnn_model_init.fit(
    training_set=training_set,
    target="label",
)

## 10. Predict (`ml_predict`)

In [11]:
datacube = tempcnn_model.predict(datacube)

## 11. Save trained model

In [12]:
tempcnn_model.save_ml_model(name="tempcnn_rondonia")

## 12. Submit job and download results

In [13]:
result = datacube.save_result(format="GeoTiff")

In [14]:
result

In [15]:
job = result.create_job(
    title="Deforestation Prediction in Rondonia",
    description="Using TempCNN model to predict deforestation in Rondonia",
)

In [16]:
job.start_and_wait()
results = job.get_results()
results.download_files("data/output-month")

0:00:00 Job '76f50e792037972003669c49210b577d': send 'start'
0:00:01 Job '76f50e792037972003669c49210b577d': running (progress N/A)
0:00:06 Job '76f50e792037972003669c49210b577d': running (progress N/A)
0:00:13 Job '76f50e792037972003669c49210b577d': running (progress N/A)
0:00:21 Job '76f50e792037972003669c49210b577d': running (progress N/A)
0:00:30 Job '76f50e792037972003669c49210b577d': running (progress N/A)
0:00:43 Job '76f50e792037972003669c49210b577d': running (progress N/A)
0:00:58 Job '76f50e792037972003669c49210b577d': running (progress N/A)
0:01:17 Job '76f50e792037972003669c49210b577d': running (progress N/A)
0:01:41 Job '76f50e792037972003669c49210b577d': running (progress N/A)
0:02:11 Job '76f50e792037972003669c49210b577d': running (progress N/A)
0:02:48 Job '76f50e792037972003669c49210b577d': running (progress N/A)
0:03:34 Job '76f50e792037972003669c49210b577d': running (progress N/A)
0:04:33 Job '76f50e792037972003669c49210b577d': running (progress N/A)
0:05:33 Job '76f

[PosixPath('data/output-month/SENTINEL-2_MSI_20LMM_class_2022-01-05.tif'),
 PosixPath('data/output-month/SENTINEL-2_MSI_20LMN_class_2022-01-05.tif'),
 PosixPath('data/output-month/SENTINEL-2_MSI_20LNM_class_2022-01-05.tif'),
 PosixPath('data/output-month/SENTINEL-2_MSI_20LNN_class_2022-01-05.tif'),
 PosixPath('data/output-month/job-results.json')]